<!-- NVN-HEADER -->
# Coarse cross-subject probe — is *amplitude* recoverable?

**Status:** superseded by `NeuroVascNet_CrossSubj_TrialLevel_Verification.ipynb`.

## Purpose
First test of predicting response **amplitude** (one number per window) instead of the full
waveform, using simple Ridge regression rather than the deep model.

## What we did
Single 30% hold-out split, Ridge on EEG features -> fNIRS/fMRI amplitude.

## What we achieved
- Established amplitude as a target worth pursuing, and produced the first non-zero
  cross-subject number (fNIRS r = 0.21).

## What we failed to achieve
- **The 0.21 was fragile.** Proper leave-one-subject-out later dropped it to ~0.01 — the
  single-split protocol was too optimistic. Read the TrialLevel notebook instead.

> **Retrospective (data-quality audit, 2026-07):** this notebook trained against the **HEFMI** fNIRS target, later shown to be **systemic-dominated** (HbO/HbR = **+0.898**, should be negative) with split-half reliability of only **0.127** after the pipeline's bandpass -> global-signal regression -> z-score chain. GSR alone removes ~83% of the evoked response. **The negative results below are confounded: they measure the preprocessing, not EEG.** See `NeuroVascNet_Data_Quality_Audit.md` sections 2.0-2.0c.

# NeuroVascNet - Coarse Cross-Subject Prediction Experiment

**Goal:** the full hemodynamic *waveform* is not recoverable from EEG across subjects (Pearson ~ 0).
This notebook asks an easier question:

> **Can we predict a COARSE SUMMARY (the *amplitude* of the hemodynamic response) - instead of the full signal - ACROSS subjects?**

**Why this might work where the waveform fails:**
- Fewer numbers to predict (e.g. 48 region-amplitudes vs 48x15 waveform points) -> far less noise.
- Averaging cancels random noise but keeps consistent signal.
- Amplitude-level coupling is what real EEG<->BOLD/fNIRS studies actually find (band-power -> response size).

**The one thing we must NOT do:** z-score each window over time (that deletes amplitude -> the summary becomes 0).
So here we build targets from the **raw cached hemo** and keep amplitude, then standardize only across windows.

**Test:** cross-subject train/test (train on some people, test on held-out people) + a **shuffled null**.
A result only counts if it **beats the null** (real EEG<->hemo pairing better than random pairing).

In [1]:
# CELL 1  Imports + load the RAW cached data (amplitude preserved)
import glob, os, numpy as np
from scipy.signal import butter, sosfiltfilt, hilbert
from sklearn.linear_model import Ridge

def load_raw(pat):
    f = sorted(glob.glob('neurovascnet_outputs/**/'+pat+'.npz', recursive=True))[0]
    z = np.load(f)
    return {'eeg': z['eeg'], 'hemo': z['hemo'], 'sid': z['subject_ids'], 'file': os.path.basename(f)}

fnirs = load_raw('fnirs_train_w30_e32*')       # HEFMI: eeg(N,32,6000) hemo(N,180,300)
fmri  = load_raw('ds002336_train_w30_e32*')    # ds002336: eeg(N,32,6000) hemo(N,48,15)
for nm,d in [('fNIRS',fnirs),('fMRI',fmri)]:
    print(nm, ': eeg', d['eeg'].shape, ' hemo', d['hemo'].shape,
          ' subjects', len(np.unique(d['sid'])), ' [', d['file'], ']')

fNIRS : eeg (1230, 32, 6000)  hemo (1230, 180, 300)  subjects 14  [ fnirs_train_w30_e32_r48.npz ]
fMRI : eeg (376, 32, 6000)  hemo (376, 48, 15)  subjects 8  [ ds002336_train_w30_e32_r48.npz ]


In [2]:
# CELL 2  EEG features = band-power (mean power per electrode per band)
# why: hemodynamics couples to how much alpha/beta/... POWER there is, not the raw squiggle.
BANDS = [(4,8),(8,13),(13,30),(30,45)]   # theta, alpha, beta, gamma
EEG_FS = 200

def bandpower_features(eeg, fs=EEG_FS, bands=BANDS, chunk=256):
    # (N,C,T) -> (N, C*len(bands)) : average band-power per electrode per band.
    sos = [butter(4,[lo,hi],btype='band',fs=fs,output='sos') for lo,hi in bands]
    out=[]
    for i in range(0,len(eeg),chunk):
        e=eeg[i:i+chunk]; f=[]
        for s in sos:
            env=np.abs(hilbert(sosfiltfilt(s,e,axis=-1),axis=-1))   # power envelope
            f.append(env.mean(axis=-1))                             # (n, C) mean over time
        out.append(np.concatenate(f,axis=1))
    return np.concatenate(out,axis=0).astype(np.float32)

Xf = bandpower_features(fnirs['eeg']); Xm = bandpower_features(fmri['eeg'])
print("fNIRS EEG features:", Xf.shape, " | fMRI EEG features:", Xm.shape)

fNIRS EEG features: (1230, 128)  | fMRI EEG features: (376, 128)


In [3]:
# CELL 3  COARSE, amplitude-preserving targets (built from RAW hemo, NOT z-scored)
# amplitude of a response = how much the signal moves over the window = std over time.
def amp_per_channel(hemo):      # (N,C,T) -> (N,C)  amplitude per channel  (medium-coarse)
    return hemo.std(axis=-1).astype(np.float32)
def amp_overall(hemo):          # (N,C,T) -> (N,1)  single overall activation level (coarsest)
    return hemo.std(axis=(1,2)).reshape(-1,1).astype(np.float32)

targets = {
    'fNIRS overall level (1 number)':    (Xf, amp_overall(fnirs['hemo']),    fnirs['sid']),
    'fNIRS per-channel amplitude (180)': (Xf, amp_per_channel(fnirs['hemo']), fnirs['sid']),
    'fMRI overall level (1 number)':     (Xm, amp_overall(fmri['hemo']),     fmri['sid']),
    'fMRI per-ROI amplitude (48)':       (Xm, amp_per_channel(fmri['hemo']),  fmri['sid']),
}
for k,(X,Y,s) in targets.items():
    print(k.ljust(36), 'X', X.shape, ' Y', Y.shape, ' target-std-across-windows=%.4f'%Y.std(0).mean())

fNIRS overall level (1 number)       X (1230, 128)  Y (1230, 1)  target-std-across-windows=0.0000
fNIRS per-channel amplitude (180)    X (1230, 128)  Y (1230, 180)  target-std-across-windows=0.0000
fMRI overall level (1 number)        X (376, 128)  Y (376, 1)  target-std-across-windows=0.2708
fMRI per-ROI amplitude (48)          X (376, 128)  Y (376, 48)  target-std-across-windows=0.3747


In [4]:
# CELL 4  Cross-subject evaluation with shuffled NULL
# train on some subjects, predict HELD-OUT subjects. Correlate predicted vs true amplitude
# across test windows (per target-column), average. Null = shuffle test targets.
def col_corr(P, Y):
    Pc=P-P.mean(0); Yc=Y-Y.mean(0)
    pv=(Pc*Pc).sum(0); yv=(Yc*Yc).sum(0); ok=(pv>1e-8)&(yv>1e-8)
    r=np.zeros(Y.shape[1]); r[ok]=(Pc*Yc).sum(0)[ok]/np.sqrt(pv[ok]*yv[ok]); return r, ok

def cross_subject(X, Y, sid, test_frac=0.30, alpha=10.0, n_null=300, seed=0):
    rng=np.random.default_rng(seed); subs=np.unique(sid)
    n_test=max(1,int(round(len(subs)*test_frac)))
    test=set(subs[-n_test:]); te=np.isin(sid,list(test)); tr=~te
    mu,sd=X[tr].mean(0),X[tr].std(0)+1e-6
    Xtr,Xte=(X[tr]-mu)/sd,(X[te]-mu)/sd
    ym,ys=Y[tr].mean(0),Y[tr].std(0)+1e-6
    Ytr,Yte=(Y[tr]-ym)/ys,(Y[te]-ym)/ys
    P=Ridge(alpha=alpha).fit(Xtr,Ytr).predict(Xte).reshape(len(Yte),-1)
    r,ok=col_corr(P,Yte); real=float(r[ok].mean()) if ok.any() else float('nan')
    nulls=np.array([col_corr(P,Yte[rng.permutation(len(Yte))])[0][ok].mean() for _ in range(n_null)])
    nci=(float(np.percentile(nulls,2.5)),float(np.percentile(nulls,97.5)))
    p=float((nulls>=real).mean())
    verdict='REAL (beats null)' if (real>nci[1] and p<0.05) else '~0 (within null)'
    return dict(real=real, null=float(nulls.mean()), nci=nci, p=p,
                n_train=int(tr.sum()), n_test=int(te.sum()), n_test_subj=n_test, verdict=verdict)

In [5]:
# CELL 5  RUN: coarse cross-subject prediction, all target granularities
print('target'.ljust(38), 'cross-subj r'.rjust(12), 'null'.rjust(8), 'p'.rjust(6), '  verdict')
print('-'*84)
for k,(X,Y,s) in targets.items():
    R=cross_subject(X,Y,s)
    print(k.ljust(38), ('%+.4f'%R['real']).rjust(12), ('%+.4f'%R['null']).rjust(8),
          ('%.3f'%R['p']).rjust(6), '  '+R['verdict'],
          '  (train %d / test %d win, %d held-out subj)'%(R['n_train'],R['n_test'],R['n_test_subj']))
print('-'*84)
print('READ: coarse (fewer-number) targets have the best shot cross-subject.')
print('      If ANY row = REAL, you have a cross-subject positive on the response AMPLITUDE.')

target                                 cross-subj r     null      p   verdict
------------------------------------------------------------------------------------
fNIRS overall level (1 number)              +0.2107  -0.0055  0.000   REAL (beats null)   (train 870 / test 360 win, 4 held-out subj)
fNIRS per-channel amplitude (180)           +0.0720  -0.0012  0.000   REAL (beats null)   (train 870 / test 360 win, 4 held-out subj)
fMRI overall level (1 number)               -0.0732  +0.0060  0.773   ~0 (within null)   (train 282 / test 94 win, 2 held-out subj)
fMRI per-ROI amplitude (48)                 -0.0396  +0.0035  0.770   ~0 (within null)   (train 282 / test 94 win, 2 held-out subj)
------------------------------------------------------------------------------------
READ: coarse (fewer-number) targets have the best shot cross-subject.
      If ANY row = REAL, you have a cross-subject positive on the response AMPLITUDE.


## How to read this

- **`cross-subj r`** = correlation between predicted and true response **amplitude**, on **held-out subjects**.
- **`null`** = same thing with the EEG<->hemo pairing scrambled (pure chance).
- **`REAL (beats null)`** = the amplitude *is* predictable across subjects -> **a genuine cross-subject result** (even if the full waveform isn't).
- **`~0 (within null)`** = even the coarse amplitude doesn't transfer -> the observability wall is total for this target.

**If a coarse target is REAL**, that becomes the cross-subject claim: *"EEG predicts the hemodynamic response amplitude across subjects, though not the full waveform."*

**Next levers if borderline:** per-region grouping (coarser than per-channel), motor-only ROIs (C3/C4-driven), a small MLP instead of Ridge, and more subjects (swap in the Multi-joint fNIRS dataset - 32/32 electrode overlap).

In [6]:
# CELL 6  DOMAIN ADAPTATION: Wasserstein source-selection + Deep CORAL (from the ICH paper)
# why: plain cross-subject fNIRS ~ 0 because of inter-subject variability. CORAL aligns each
# training subject's feature covariance to the TEST subject's (uses target EEG only, NOT labels
# = valid unsupervised domain adaptation), which unlocks cross-subject transfer.
from scipy.stats import wasserstein_distance
def _sqrtm(C):
    w,V=np.linalg.eigh(C); w=np.clip(w,1e-10,None); return (V*np.sqrt(w))@V.T
def _invsqrtm(C):
    w,V=np.linalg.eigh(C); w=np.clip(w,1e-10,None); return (V/np.sqrt(w))@V.T
def coral(Xs,Xt,eps=1e-1):
    ms,mt=Xs.mean(0),Xt.mean(0)
    Cs=np.cov(Xs-ms,rowvar=False)+eps*np.eye(Xs.shape[1])
    Ct=np.cov(Xt-mt,rowvar=False)+eps*np.eye(Xs.shape[1])
    return (Xs-ms)@(_invsqrtm(Cs)@_sqrtm(Ct))+mt
def _wdist(A,B): return np.mean([wasserstein_distance(A[:,j],B[:,j]) for j in range(A.shape[1])])

def loso_da(X,Y,sid,mode='coral',k=6,alpha=10.,n_null=300,seed=0):
    rng=np.random.default_rng(seed);subs=np.unique(sid);preds=[];trues=[]
    for s in subs:
        te=sid==s
        if te.sum()<3: continue
        if mode in ('select','both'):
            pool=[u for u in subs if u!=s]; d={u:_wdist(X[sid==u],X[te]) for u in pool}
            tr=np.isin(sid,sorted(d,key=d.get)[:k])
        else: tr=~te
        Xtr,Xte=X[tr].copy(),X[te].copy()
        if mode in ('coral','both'): Xtr=coral(Xtr,Xte)
        mu,sd=Xtr.mean(0),Xtr.std(0)+1e-6; ym,ys=Y[tr].mean(0),Y[tr].std(0)+1e-6
        P=Ridge(alpha=alpha).fit((Xtr-mu)/sd,(Y[tr]-ym)/ys).predict((Xte-mu)/sd)
        preds.append(P.reshape(te.sum(),-1)); trues.append(((Y[te]-ym)/ys).reshape(te.sum(),-1))
    P=np.concatenate(preds);T=np.concatenate(trues)
    def cc(P,Y):
        Pc=P-P.mean(0);Yc=Y-Y.mean(0);pv=(Pc*Pc).sum(0);yv=(Yc*Yc).sum(0);ok=(pv>1e-8)&(yv>1e-8)
        r=np.zeros(Y.shape[1]);r[ok]=(Pc*Yc).sum(0)[ok]/np.sqrt(pv[ok]*yv[ok]);return r,ok
    r,ok=cc(P,T);real=float(r[ok].mean())
    nl=np.array([cc(P,T[rng.permutation(len(T))])[0][ok].mean() for _ in range(n_null)])
    p=float((nl>=real).mean());v='REAL' if (real>np.percentile(nl,97.5) and p<0.05) else '~0'
    return real,float(nl.mean()),p,v

Yall=fnirs['hemo'].std((1,2)).reshape(-1,1).astype(np.float32)
Ych =fnirs['hemo'].std(-1).astype(np.float32)
print("fNIRS cross-subject amplitude with domain adaptation (LOSO + null)")
print("%-14s%-10s%9s%8s%7s  verdict"%('target','method','r','null','p'))
for tn,Y in [('overall',Yall),('per-channel',Ych)]:
    for m in ['baseline','select','coral','both']:
        r,nu,p,v=loso_da(Xf,Y,fnirs['sid'],mode=('baseline' if m=='baseline' else m))
        if m=='baseline':
            # baseline = no DA: reuse coral path with identity by mode name trick
            r,nu,p,v=loso_da(Xf,Y,fnirs['sid'],mode='none')
        print("%-14s%-10s%+9.4f%+8.4f%7.3f  %s"%(tn,m,r,nu,p,v))


fNIRS cross-subject amplitude with domain adaptation (LOSO + null)
target        method            r    null      p  verdict
overall       baseline    +0.0123 +0.0024  0.337  ~0
overall       select      +0.1258 -0.0032  0.000  REAL
overall       coral       +0.2533 +0.0007  0.000  REAL
overall       both        +0.2278 -0.0001  0.000  REAL
per-channel   baseline    -0.0121 +0.0007  0.917  ~0
per-channel   select      +0.0294 -0.0002  0.000  REAL
per-channel   coral       +0.1030 +0.0003  0.000  REAL
per-channel   both        +0.1217 +0.0008  0.000  REAL
